In [16]:
import torch 


# input of layer 2
bs = 16 # batch size

Leye = torch.rand(bs,128, 8, 8)
print(Leye)
Reye = torch.rand(bs,128, 8, 8)
print(Reye)
FaceData = torch.rand(bs,128, 8, 8)  # currently matched with eyes......
print(FaceData)

tensor([[[[8.1329e-01, 9.8043e-01, 2.4900e-01,  ..., 3.7914e-01,
           3.0441e-01, 5.3311e-01],
          [5.3967e-01, 9.9720e-01, 1.6417e-01,  ..., 6.2011e-01,
           6.8858e-01, 8.8324e-01],
          [6.1782e-01, 2.8768e-01, 9.8886e-01,  ..., 6.2303e-01,
           2.9663e-01, 8.9251e-01],
          ...,
          [7.5977e-02, 5.6523e-01, 4.1378e-01,  ..., 4.7827e-01,
           4.8058e-02, 3.9134e-01],
          [1.7157e-01, 6.0188e-01, 4.8465e-01,  ..., 5.7554e-01,
           3.5718e-01, 1.1502e-02],
          [8.4467e-01, 5.4948e-01, 6.9289e-01,  ..., 9.4946e-01,
           2.9195e-01, 5.3210e-02]],

         [[5.8845e-01, 4.6625e-01, 8.5185e-01,  ..., 5.5059e-01,
           8.7360e-01, 7.5750e-01],
          [9.7717e-01, 9.3136e-01, 2.2619e-01,  ..., 2.3833e-02,
           7.9779e-01, 8.2218e-01],
          [5.2592e-01, 4.8483e-01, 3.7618e-01,  ..., 2.1184e-01,
           2.1715e-01, 6.0478e-01],
          ...,
          [8.4526e-01, 8.2046e-02, 1.9868e-01,  ..., 1.4916

In [ ]:
# from vit_pytorch import ViT
# https://github.com/lucidrains/vit-pytorch/blob/main/vit_pytorch/vit.py
from vit_pytorch.vit import Transformer


class TinyModel(torch.nn.Module):
    def __init__(self, gaze_dims=3):    # the gaze_dims should be defined later, according to the dataset and what we wanna predict, for instance, 3 gaze dims, PoG (x,y,z)
        super(TinyModel, self).__init__()
        # layer 1: feature extraction (to be implemented)
        
        
        # layer 2: feature fusion: concate + group  normalization
        # self.concate = torch.cat((x, x, x), 0) // not needed here, only in forward
        # total_ch = Leye[1]+Reye[1]+FaceData[1]  # total channels of the input // this is not working 
        total_ch = 384
        self.gn = torch.nn.GroupNorm(3, total_ch)     # Separate 6 channels into 3 groups, how to define a appropriate number of groups & channels?
        
        # layer 3:self-attention
        ########################################## should be replaced with the Attention class, not the whole ViT ##########################################
        # self.vit = ViT(
        #         image_size = 8,
        #         patch_size = 2,
        #         num_classes = gaze_dims,  # for instance 3 gaze dims, PoG (x,y,z)
        #         dim = 1024,
        #         depth = 6,
        #         heads = 16,
        #         mlp_dim = 2048,
        #         channels= total_ch,
        #         dropout = 0.1,
        #         emb_dropout = 0.1)
        #####################################################################################################################################################

        self.self_att = Transformer(
                dim = total_ch,   # if not using the total_ch, should project the input to the dim of the transformer first
                depth = 6,
                heads = 16,
                dim_head = total_ch//16,  #dim//heads
                mlp_dim = 2048,  # the hidden layer dim of the mlp (the hidden layer of the feedforward network, which is applied to each position (each token) separately and identically)
                # dropout = 0.
                # def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.)
                
        )

 
        # RNN layer for temporal information
        # https://pytorch.org/docs/stable/generated/torch.nn.GRU.html
        # torch.nn.GRU(input_size, hidden_size, num_layers=1, bias=True, batch_first=False, dropout=0.0, bidirectional=False, device=None, dtype=None)
        # how to define the input_size and hidden_size?
        # test num_layers param, what does it affect?
        self.rnn = torch.nn.GRU(total_ch, 512, num_layers=1, bias=True, batch_first=False, dropout=0.0, bidirectional=False, device=None, dtype=None)

    def forward(self, left_eye, right_eye, face):

        # layer2
        concate = torch.cat((left_eye, right_eye, face), 1)  # dim = 0 or 1?  only channel dim changes?
        out = self.gn(concate)
        bs, c, h, w = out.shape
        x_att = out.reshape(bs, c, h * w).transpose(1, 2)   # (bs, h*w, c) --- (bs, seq_len, features)
        x_att = self.self_att(x_att)  # output shape (bs, h*w, c)
        # h_n itself should be an input for GRU, otherwise useless, dont forget the hidden state
        out, h_n = self.rnn(x_att, h_state) # read the source, there are 2 outputs, but what is h_n here? should be the hidden state of the last layer?
        # mind the coherence of the input and output of the RNN layer 
        
        #reshape the output




        # for input frames, the temporal information should be considered, how to define the input_size and hidden_size?
        return out
        

        # set learning rate for diff layers
     


model = TinyModel(gaze_dims=3)
print(model)

output = model(Leye, Reye, FaceData)  # currently the face Data is not matched with eyes, should be matched in the future, how to match? linear padding or other methods? would it affect the performance if change the size of facedata
print("output:",output.shape)
# testing only forward pass, not backward pass


TinyModel(
  (gn): GroupNorm(3, 384, eps=1e-05, affine=True)
  (self_att): Transformer(
    (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
    (layers): ModuleList(
      (0-5): 6 x ModuleList(
        (0): Attention(
          (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
          (attend): Softmax(dim=-1)
          (dropout): Dropout(p=0.0, inplace=False)
          (to_qkv): Linear(in_features=384, out_features=1152, bias=False)
          (to_out): Sequential(
            (0): Linear(in_features=384, out_features=384, bias=True)
            (1): Dropout(p=0.0, inplace=False)
          )
        )
        (1): FeedForward(
          (net): Sequential(
            (0): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=384, out_features=2048, bias=True)
            (2): GELU(approximate='none')
            (3): Dropout(p=0.0, inplace=False)
            (4): Linear(in_features=2048, out_features=384, bias=True)
     